# PA3

In [0]:
%pip install databricks-vectorsearch
%pip install langchain_community
%pip install langgraph
dbutils.library.restartPython()
%pip install PyPDF2
%pip install langchain
%pip install pdfplumber
%pip install databricks-sdk


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Task 0: Infrastructure Provisioning

In [0]:
import os
import re
import time
from dotenv import load_dotenv
import numpy as np
from scipy.spatial.distance import cosine
import pyspark
import pandas as pd
from pyspark.sql.functions import col
from pyspark.sql.types import FloatType
from typing import Iterator
from dotenv import load_dotenv
from openai import AzureOpenAI
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import (
    ChatPromptTemplate, 
    SystemMessagePromptTemplate, 
    HumanMessagePromptTemplate
)


from databricks.vector_search.client import VectorSearchClient
from pyspark.sql.functions import udf, explode, col, pandas_udf
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

load_dotenv()

# THE ABOVE IS WHAT YOU NEED FOR THIS ASSIGNMENT

True

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS `25280065-pa3`.`default`.rag_documents;

## Part 1: RAG Implementation from Scratch

In [0]:
pdf_directory_path = "/Volumes/25280065-pa3/default/rag_documents/"
def ai_parse_document(path):
    import pdfplumber
    local_path = path.replace("dbfs:/Volumes/", "/Volumes/")
    with pdfplumber.open(local_path) as pdf:
        return "\n".join(page.extract_text() or "" for page in pdf.pages)

def ai_extract(document_path, parsed_document):
    lines = [l.strip() for l in parsed_document.split("\n") if l.strip()]
    
    # Extract Title - find first long meaningful line
    title = "Unknown"
    for line in lines:
        if len(line) > 15 and not any(c.isdigit() for c in line[:5]):
            title = line
            break

    # Extract Authors - look for lines with keywords or name patterns
    authors = "Unknown"
    for i, line in enumerate(lines):
        if any(keyword in line.lower() for keyword in ["author", "by ", "submitted", "written"]):
            authors = lines[i + 1] if i + 1 < len(lines) else line
            break
        if "," in line and all(w[0].isupper() for w in line.split() if w):
            authors = line
            break

    content_lines = []
    for line in lines:
        content_lines.append(line)

    return title, authors, content, document_path

### Task 1: Constructing the Knowledge Base

In [0]:
files_in_directory = dbutils.fs.ls(pdf_directory_path)
pdf_files = [file.path for file in files_in_directory if file.name.endswith(".pdf")]

spark.sql("CREATE SCHEMA IF NOT EXISTS parsed_data")
spark.sql("""CREATE TABLE IF NOT EXISTS parsed_data.parsed_documents (
        document_path STRING, parsed_document STRING)""")
spark.sql("TRUNCATE TABLE parsed_data.parsed_documents")

for pdf_path in pdf_files:
    parsed_text = ai_parse_document(pdf_path)
    df = spark.createDataFrame([(pdf_path, parsed_text)], ["document_path", "parsed_document"])
    df.write.mode("append").saveAsTable("parsed_data.parsed_documents")
spark.sql("CREATE SCHEMA IF NOT EXISTS knowledge_base_data")
spark.sql("DROP TABLE IF EXISTS knowledge_base_data.knowledge_base")
spark.sql("""CREATE TABLE IF NOT EXISTS knowledge_base_data.knowledge_base (
        Id BIGINT GENERATED ALWAYS AS IDENTITY,
        Title STRING, Authors STRING, Content STRING, DocumentURI STRING)
    TBLPROPERTIES (delta.enableChangeDataFeed = true)""")
spark.sql("TRUNCATE TABLE knowledge_base_data.knowledge_base")

for _, row in spark.sql("SELECT * FROM parsed_data.parsed_documents").toPandas().iterrows():
    title, authors, content, doc_uri = ai_extract(row["document_path"], row["parsed_document"])
    df = spark.createDataFrame([(title, authors, content, doc_uri)], ["Title", "Authors", "Content", "DocumentURI"])
    df.write.mode("append").saveAsTable("knowledge_base_data.knowledge_base")
# Verify parsed_documents
display(spark.sql("SELECT * FROM parsed_data.parsed_documents"))

# Verify knowledge_base
display(spark.sql("SELECT * FROM knowledge_base_data.knowledge_base"))

document_path parsed_document dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf 5202
yaM
71
]IA.sc[
1v56021.5052:viXra
Demystifying and Enhancing the Efficiency of Large
Language Model Based Search Agents
TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,
GangWang1,XiaoguangLiu1
1NankaiUniversity
2UniversityofIllinoisUrbana-Champaign
{yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn
bowenj4@illinois.edu
Abstract
LargeLanguageModel(LLM)-basedsearchagentshaveshownremarkableca-
pabilitiesinsolvingcomplextasksbydynamicallydecomposingproblemsand
addressing them through interleaved reasoning and retrieval. However, this in-
terleaved paradigm introduces substantial efficiency bottlenecks. First, we ob-
serve that both highly accurate and overly approximate retrieval methods de-
gradesystemefficiency: exactsearchincurssignificantretrievaloverhead,while
coarse retrieval requires additional reasoning steps during generation. Second,
we identify inefficiencies in system design, including improper scheduling and
frequentretrievalstalls,whichleadtocascadinglatency—whereevenminorde-
laysinretrievalamplifyend-to-endinferencetime. Toaddressthesechallenges,
weintroduceSearchAgent-X,ahigh-efficiencyinferenceframeworkforLLM-
basedsearchagents. SearchAgent-Xleverageshigh-recallapproximateretrieval
and incorporates two key techniques: priority-aware scheduling and non-stall
retrieval. ExtensiveexperimentsdemonstratethatSearchAgent-Xconsistently
outperforms state-of-the-art systems such as vLLM and HNSW-based retrieval
across diverse tasks, achieving up to 3.4× higher throughput and 5× lower la-
tency,withoutcompromisinggenerationquality. SearchAgent-Xisavailableat
https://github.com/tiannuo-yang/SearchAgent-X.
1 Introduction
TraditionalRetrieval-AugmentedGeneration(RAG)typicallyusesasequentialretrieve-then-generate
approach [1–8], which limits dynamic interaction with knowledge bases. Recent advancements
have ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong
reasoningcapabilitiesofLargeLanguageModels(LLMs),allowingforthedynamicandadaptive
interleavingofreasoningstepswithretrievalcallsthroughoutthegenerationprocess. Insteadofa
fixedpipeline,searchagentscandecidewhenandwhattoretrievebasedonLLM’songoingreasoning,
leadingtosignificantimprovementsinthequalityanddepthofthegeneratedresponses. Leveraging
post-trainingtechniquessimilartoDeepSeek-R1,somepioneeringmodelscanevenautonomously
initiateretrievalactionsduringreasoningwithoutintermediatesupervision[12–14].
However, the improved generation quality achieved by search agents often comes at the cost of
efficiency—anoverheadthatisnontrivialinpracticaldeployments. Inreasoning-with-searchsce-
narios,achievinglow-latencyresponsesiscriticalforensuringaseamlessuserexperience[17,18].
Moreover,duringpost-trainingofLLM-basedsearchagents,efficientmodelrolloutsoverlarge-scale
∗EqualContribution
Preprint.Underreview.
trainingcorporaareessentialtosupportscalablelearning. Whilerecentsystemsincorporateadvanced
inferenceoptimizations—suchassequenceconcatenation[12–14]andprefixcaching[18–20]—these
techniquesarenotspecificallydesignedtoaddresstheuniquecomputationalchallengesposedbythe
tightinterleavingofmulti-stepreasoninganddynamicretrieval.
Tothisend,wefirstconductasystematicanalysisoftheefficiencyfactorsgoverningLLM-based
searchagents,uncoveringinsightsthatdivergefromtheunderstandingofnaiveRAG.Ourin-depth
analysisrevealstwokeyobservations: First,wedemonstrateanon-monotonicrelationshipbetween
retrievalaccuracyandend-to-endefficiency.Bothexcessivelyhigh(e.g.,exactsearch)andexcessively
lowretrievalrecalldegradeoverallefficiency. Whileaimingforperfectrecallincursunnecessary
computationaloverheadintheretrievalphase,lowrecallnecessitatesmoreretrievaliterationsand
longerreasoningpathsbytheLLMtocompensate(asshowninFigure1). Thishighlightsthatsearch
agentsystemsbenefitfromhigh-recallapproximatesearchthateffectivelysupportsreasoningwithout
unnecessaryret

Id,Title,Authors,Content,DocumentURI
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,","Demystifying and Enhancing the Efficiency of Large TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1, {yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn LargeLanguageModel(LLM)-basedsearchagentshaveshownremarkableca- pabilitiesinsolvingcomplextasksbydynamicallydecomposingproblemsand addressing them through interleaved reasoning and retrieval. However, this in- terleaved paradigm introduces substantial efficiency bottlenecks. First, we ob- serve that both highly accurate and overly approximate retrieval methods de- gradesystemefficiency: exactsearchincurssignificantretrievaloverhead,while coarse retrieval requires additional reasoning steps during generation. Second, we identify inefficiencies in system design, including improper scheduling and frequentretrievalstalls,whichleadtocascadinglatency—whereevenminorde- laysinretrievalamplifyend-to-endinferencetime. Toaddressthesechallenges, weintroduceSearchAgent-X,ahigh-efficiencyinferenceframeworkforLLM- basedsearchagents. SearchAgent-Xleverageshigh-recallapproximateretrieval and incorporates two key techniques: priority-aware scheduling and non-stall retrieval. ExtensiveexperimentsdemonstratethatSearchAgent-Xconsistently outperforms state-of-the-art systems such as vLLM and HNSW-based retrieval across diverse tasks, achieving up to 3.4× higher throughput and 5× lower la- tency,withoutcompromisinggenerationquality. SearchAgent-Xisavailableat https://github.com/tiannuo-yang/SearchAgent-X. TraditionalRetrieval-AugmentedGeneration(RAG)typicallyusesasequentialretrieve-then-generate approach [1–8], which limits dynamic interaction with knowledge bases. Recent advancements have ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong reasoningcapabilitiesofLargeLanguageModels(LLMs),allowingforthedynamicandadaptive interleavingofreasoningstepswithretrievalcallsthroughoutthegenerationprocess. Insteadofa fixedpipeline,searchagentscandecidewhenandwhattoretrievebasedonLLM’songoingreasoning,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf


### Task 2: Implement Chunking Strategies

In [0]:
class MyTextSplitter():
    def __init__(self, chunk_size=512, chunk_overlap=0):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def split_text(self, text):
        chunks = []
        start = 0
        while start < len(text):
            end = start + self.chunk_size
            chunks.append(text[start:end])
            start += self.chunk_size - self.chunk_overlap
        return chunks


class MySemanticSplitter:
    def __init__(self, client, model_name, distance_threshold=0.3, buffer_size=1):
        self.client = client
        self.model_name = model_name
        self.distance_threshold = distance_threshold
        self.buffer_size = buffer_size
    
    def _combine_sentences(self, sentences, buffer_size):
        combined = []
        for i in range(len(sentences)):
            start = max(0, i - buffer_size)
            end = min(len(sentences), i + buffer_size + 1)
            combined.append(" ".join(sentences[start:end]))
        return combined

    @staticmethod
    def cosine_similarity(embedding1, embedding2):
        a = np.array(embedding1)
        b = np.array(embedding2)
        cosine_similarities = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
        return cosine_similarities

    def _get_embedding(self, text):
        response = self.client.embeddings.create(
            input=text,
            model=self.model_name
        )
        embedding = response.data[0].embedding
        return embedding

    def split_text(self, text):
        chunks = []
        sentences = [s.strip() for s in text.split(".") if s.strip()]
        combined = self._combine_sentences(sentences, self.buffer_size)
        embeddings = [self._get_embedding(s) for s in combined]
        current_chunk = sentences[0]
        for i in range(1, len(sentences)):
            similarity = self.cosine_similarity(embeddings[i - 1], embeddings[i])
            distance = 1 - similarity
            if distance > self.distance_threshold:
                chunks.append(current_chunk)
                current_chunk = sentences[i]
            else:
                current_chunk += ". " + sentences[i]
        chunks.append(current_chunk)
        return chunks
    

In [0]:
### SANITY CHECK ###

# Sample Research Paper
sample_document = (
    "Abstract: The exponential growth in data has created a need for platforms capable of storing both structured & unstructured data, effectively processing the data, analyzing, and creating machine learning models. Traditional data lakes and warehouses often lack the flexibility and performance to provide these capabilities. So, the new Lakehouse paradigm is introduced. Databricks is an implementation of the Lakehouse paradigm that is cloud-native and built upon Apache Spark. However, there is a lack of substantial academic work describing the ecosystem. This paper presents a comprehensive description of the Databricks ecosystem, showing it both as an architecture and as a platform already in use. We will go through the main components of Databricks architecture, including Delta Lake, Unity Catalog, cluster management, Azure integrations, and discuss their roles in creating secure, scalable, and cost-effective data engineering workflows. We also present an experiment that is designed to bridge the gap between theory and practice by demonstrating the ingestion of data, feature engineering, and basic fraud detection. We utilized a synthetic dataset of financial transactions. The experimental procedure and metrics such as query latency, processing throughput, speed, and performance results are described in detail offering a reproducible benchmark for evaluating similar workloads in Databricks. This work is designed to function as a technical resource for individuals in industry, data engineers, and researchers who are interested in working with the Databricks environment for large-scale analytics. Keywords: Databricks, Lakehouse, Delta Lake, Data Engineering, PySpark, Unity Catalog, MLflow, Azure Synapse, Data Factory, Power BI, Real-time Analytics, Cloud Data Platforms, Fraud Detection, Performance Benchmarking, Industry Use Case. 1. Introduction These days organizations have high volume of data in the form of structured, semistructured and unstructured data. They need platforms that can integrate storage, analytics and machine learning at scale in order to gain value from their data. Conventional data warehouses store structured data for analytical needs (such as reporting through Power BI), but lack flexibility. Data lakes, on the other hand, can store semistructured and unstructured data using batch data processing, but do not provide ACID guarantees. The new Lakehouse model provides the best of both worlds to store and..."
)

# --- Test 1: The Normal Text Splitter (Fixed Size) ---
print("====== FIXED-SIZE CHUNKER ======")
normal_splitter = MyTextSplitter(chunk_size=512, chunk_overlap=50)
normal_chunks = normal_splitter.split_text(sample_document)

for i, chunk in enumerate(normal_chunks[:3]):
    print(f"\n--- Chunk {i+1} (Length: {len(chunk)}) ---")
    print(chunk)


# --- Test 2: The Semantic Splitter (Embedding-Based) ---
print("\n\n====== SEMANTIC CHUNKER ======")
client = AzureOpenAI(
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
)
semantic_splitter = MySemanticSplitter(
    client=client, 
    model_name=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"], 
    distance_threshold=0.15, 
    buffer_size=1
)
semantic_chunks = semantic_splitter.split_text(sample_document)

for i, chunk in enumerate(semantic_chunks):
    print(f"\n--- Chunk {i+1} (Length: {len(chunk)}) ---")
    print(chunk)

====== FIXED-SIZE CHUNKER ======

--- Chunk 1 (Length: 512) ---
Abstract: The exponential growth in data has created a need for platforms capable of storing both structured & unstructured data, effectively processing the data, analyzing, and creating machine learning models. Traditional data lakes and warehouses often lack the flexibility and performance to provide these capabilities. So, the new Lakehouse paradigm is introduced. Databricks is an implementation of the Lakehouse paradigm that is cloud-native and built upon Apache Spark. However, there is a lack of substa

--- Chunk 2 (Length: 512) ---
n Apache Spark. However, there is a lack of substantial academic work describing the ecosystem. This paper presents a comprehensive description of the Databricks ecosystem, showing it both as an architecture and as a platform already in use. We will go through the main components of Databricks architecture, including Delta Lake, Unity Catalog, cluster management, Azure integrations, and di

In [0]:
fixed_splitter = MyTextSplitter(chunk_size=512, chunk_overlap=50)
spark.sql("DROP TABLE IF EXISTS processed_data.fixed_chunks")
spark.sql("CREATE SCHEMA IF NOT EXISTS processed_data")
spark.sql("""
CREATE TABLE IF NOT EXISTS processed_data.fixed_chunks (
    chunk_id BIGINT GENERATED ALWAYS AS IDENTITY,
    document_id BIGINT,
    title STRING,
    author STRING,
    document_uri STRING,
    content STRING,
    chunk_text STRING
)
""")
spark.sql("TRUNCATE TABLE processed_data.fixed_chunks")
# 1. Load the knowledge base table
kb_df = spark.sql("""
SELECT Id, Title, Authors, DocumentURI, Content
FROM knowledge_base_data.knowledge_base
""")

# 2. Split the content into chunks
all_chunks = []
for row in kb_df.toLocalIterator():
    doc_id = row["Id"]
    title = row["Title"]
    author = row["Authors"]
    document_uri = row["DocumentURI"]
    content = row["Content"]
    chunks = fixed_splitter.split_text(content)
    for chunk in chunks:
        all_chunks.append((doc_id, title, author, document_uri, content, chunk))

# 3. Write the chunks to a new table
df = spark.createDataFrame(all_chunks, ["document_id", "title", "author", "document_uri", "content", "chunk_text"])
df.write.option("mergeSchema", "true").mode("append").saveAsTable("processed_data.fixed_chunks")
display(spark.sql("""
SELECT document_id, title, author, document_uri, SUBSTRING(chunk_text, 1, 120) AS context
FROM processed_data.fixed_chunks
"""))

document_id,title,author,document_uri,context
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,"Demystifying and Enhancing the Efficiency of Large TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1, {yangtn,yaozb"
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,that both highly accurate and overly approximate retrieval methods de- gradesystemefficiency: exactsearchincurssignific
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,"archAgent-X,ahigh-efficiencyinferenceframeworkforLLM- basedsearchagents. SearchAgent-Xleverageshigh-recallapproximateret"
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,onquality. SearchAgent-Xisavailableat https://github.com/tiannuo-yang/SearchAgent-X. TraditionalRetrieval-AugmentedGener
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,"ngstepswithretrievalcallsthroughoutthegenerationprocess. Insteadofa fixedpipeline,searchagentscandecidewhenandwhattoretr"


In [0]:
client = AzureOpenAI(
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
)

semantic_splitter = MySemanticSplitter(
    client=client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    distance_threshold=0.15
)
spark.sql("DROP TABLE IF EXISTS processed_data.semantic_chunks")
spark.sql("CREATE SCHEMA IF NOT EXISTS processed_data")
spark.sql("""
CREATE TABLE IF NOT EXISTS processed_data.semantic_chunks (
    chunk_id BIGINT GENERATED ALWAYS AS IDENTITY,
    document_id BIGINT,           
    title STRING,
    author STRING,          
    document_uri STRING,    
    content STRING,
    chunk_text STRING
)
""")
spark.sql("TRUNCATE TABLE processed_data.semantic_chunks")

# 1. Load the knowledge base table
kb_df = spark.sql("""
SELECT Id, Title, Authors, DocumentURI, Content
FROM knowledge_base_data.knowledge_base
""")
# 2. Split the content into chunks
all_chunks = []
for row in kb_df.toLocalIterator():
    doc_id = row["Id"]
    title = row["Title"]
    author = row["Authors"]
    document_uri = row["DocumentURI"]
    content = row["Content"]
    chunks = semantic_splitter.split_text(content)
    for chunk in chunks:
        all_chunks.append((doc_id, title, author, document_uri, content, chunk))
# 3. Write the chunks to a new table
df = spark.createDataFrame(all_chunks, ["document_id", "title", "author", "document_uri", "content", "chunk_text"])
df.write.option("mergeSchema", "true").mode("append").saveAsTable("processed_data.semantic_chunks")
display(spark.sql("""
SELECT document_id, title, author, document_uri, SUBSTRING(chunk_text, 1, 120) AS context
FROM processed_data.semantic_chunks
"""))

document_id,title,author,document_uri,context
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,"Demystifying and Enhancing the Efficiency of Large TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1, {yangtn,yaozb"
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,edu. cn LargeLanguageModel(LLM)-basedsearchagentshaveshownremarkableca- pabilitiesinsolvingcomplextasksbydynamicallydeco
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,"First, we ob- serve that both highly accurate and overly approximate retrieval methods de- gradesystemefficiency: exacts"
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,"Second, we identify inefficiencies in system design, including improper scheduling and frequentretrievalstalls,whichlead"
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,SearchAgent-Xleverageshigh-recallapproximateretrieval and incorporates two key techniques: priority-aware scheduling and
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,ExtensiveexperimentsdemonstratethatSearchAgent-Xconsistently outperforms state-of-the-art systems such as vLLM and HNSW-
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,"4× higher throughput and 5× lower la- tency,withoutcompromisinggenerationquality"
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,SearchAgent-Xisavailableat https://github
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,com/tiannuo-yang/SearchAgent-X. TraditionalRetrieval-AugmentedGeneration(RAG)typicallyusesasequentialretrieve-then-gener
1,Demystifying and Enhancing the Efficiency of Large,"TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,",dbfs:/Volumes/25280065-pa3/default/rag_documents/rag_research_paper.pdf,"0, known as Search Agents [9–16]"


### Task 3: Generating Vector Embeddings

In [0]:
class GenerateEmbeddings:
    def __init__(self, client, model_name, chunk_table_name):
        self.client = client
        self.model_name = model_name
        self.chunk_table_name = chunk_table_name

    def embed(self):
        chunk_df = spark.sql(f"SELECT document_id, chunk_text FROM {self.chunk_table_name}")
        records = []
        for row in chunk_df.toLocalIterator():
            chunk_text = row["chunk_text"]
            response = self.client.embeddings.create(input=chunk_text, model=self.model_name)
            embedding = [float(x) for x in response.data[0].embedding]  # cast to float
            records.append((chunk_text, embedding))

        schema = StructType([
            StructField("chunk_text", StringType(), True),
            StructField("embedding", ArrayType(FloatType()), True) 
        ])
        self.embeddings = spark.createDataFrame(records, schema=schema)

    def save_to_table(self, table_name):
        self.embeddings.write.format("delta").mode("overwrite").saveAsTable(table_name)


In [0]:
spark.sql("DROP TABLE IF EXISTS processed_data.fixed_embeddings")
spark.sql("""
CREATE TABLE IF NOT EXISTS processed_data.fixed_embeddings (
    chunk_text STRING,
    embedding ARRAY<FLOAT>
)
""")
client = AzureOpenAI(
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
)
fixed_embedding_generator = GenerateEmbeddings(
    client=client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    chunk_table_name="processed_data.fixed_chunks"
)
fixed_embedding_generator.embed()
fixed_embedding_generator.embeddings.write.option("overwriteSchema", "true").mode("append").saveAsTable("processed_data.fixed_embeddings")
display(spark.sql("""
SELECT chunk_text, embedding FROM processed_data.fixed_embeddings
"""))

chunk_text embedding Demystifying and Enhancing the Efficiency of Large
TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,
{yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn
LargeLanguageModel(LLM)-basedsearchagentshaveshownremarkableca-
pabilitiesinsolvingcomplextasksbydynamicallydecomposingproblemsand
addressing them through interleaved reasoning and retrieval. However, this in-
terleaved paradigm introduces substantial efficiency bottlenecks. First, we ob-
serve that both highly accurate and overly approximate List(0.012123311, 0.003124059, 0.057050876, 0.021695035, -0.009774542, 0.04346858, -0.005979874, 0.030828409, -0.036324136, 0.04556219, 0.043102197, -0.051371958, -0.042160075, -6.706095E-4, -0.0114298025, -0.015806757, 0.005904635, -0.047708143, 0.037894342, 0.0030815324, 0.059301507, -0.025542043, 0.0104157105, 0.009270768, 0.015309524, -0.029415222, 0.014131868, 0.05192153, 0.023409178, -0.04493411, 0.053177696, -0.03687371, 7.3767046E-4, 0.015100163, -0.010350285, 0.06484957, 0.036402646, 0.01728537, -0.006617771, 0.013804742, 0.006009315, 0.017586324, -0.01325517, -0.0129149575, 0.016788136, 0.042133905, -0.006235033, 0.03781583, -0.024429813, 0.014786121, 0.0026137414, 0.019941637, -0.016395584, -0.048650265, -0.034256697, -0.044750918, 0.011560653, 8.2681247E-4, 0.046164103, 0.02459992, -0.041008588, -0.032084577, -0.011868153, 0.041793693, -0.060662355, -0.03297436, -0.03103777, -0.0029179691, -0.02894416, -0.01237847, -0.041924544, -2.4023358E-4, 0.04723708, -0.010566189, -0.0010999631, -0.003372675, 0.011802727, 0.07118274, -0.007386518, 0.0071640722, 0.010075498, -0.007667847, 0.02473077, -0.06186618, -0.010232519, -0.018620046, 0.004690995, 0.0054564714, -0.050456006, -0.07097338, -0.020150997, 0.042945176, 0.00594389, -0.012653256, 0.047838993, 0.025856085, -0.00873428, 0.022885775, 0.064273834, 0.04323305, -0.011547568, -0.0103306575, 0.04553602, -0.036219455, 0.003431558, -0.017560154, -0.022244608, -0.015034738, -0.031377982, 0.009401618, -0.08840269, -0.0017092364, -0.027923524, -0.034152016, -0.006117267, -0.056527473, -0.024796195, 0.043023687, -0.021210888, 0.054172162, 0.0062808306, 0.010140924, -0.009800713, 0.0036016637, 7.4911985E-4, -0.009146459, 0.04375645, -0.03888881, 0.013451445, 0.005119531, 0.030462027, -0.0046975375, 0.007994974, -0.0034773557, -0.06914148, -0.0051031746, -0.06626276, 0.016042288, -6.804233E-4, 0.028132886, 0.023827901, -0.027269272, 0.005688731, 0.03525116, -0.052811317, 0.026654273, -0.0020150999, -0.015545055, -0.017010583, 0.0070201363, -0.007667847, -0.09028694, -0.0459024, 0.006542532, -0.019064937, -0.032267768, -0.0054924553, 0.00589155, 0.044463046, -0.039647743, 0.03334074, 0.019863127, 0.008197792, 0.015100163, 0.027818846, 0.03310521, -0.028316077, 0.023317583, -0.056841515, -0.032607976, -0.013660806, -0.071601465, -0.0052700094, -0.026327148, 0.017743345, -0.037920512, -0.0063560694, -0.030095646, -0.033497762, -0.029912455, -0.008112739, 0.030409688, -0.0021230516, -0.0061826925, -0.0128429895, -0.03768498, 0.0030978888, 0.050429836, -0.00560695, 0.014864632, -0.02162961, 0.06772829, 0.07039764, 0.022911945, -0.008060399, 0.027243102, 0.042290926, -0.008485664, -0.02274184, 0.014655271, 0.044803258, 0.015100163, 0.002922876, -0.029415222, -0.0057279863, -0.017403135, 0.015623566, 0.026091617, -0.015846012, -0.0494092, 0.0012717046, -0.008995981, 0.04582389, -0.021498758, -0.017769516, -0.009061406, -0.04155816, 0.01545346, -0.0151132485, 0.010252147, -0.01943132, -0.0012700689, 0.014131868, -0.0020739825, -0.04726325, 0.022781095, 0.033497762, -0.058725763, 0.022885775, -0.00310116, 0.060871713, -0.011711132, -0.00979417, 0.002767491, 0.02367088, -0.023867156, 0.017429305, -0.0070070513, -0.0041316086, -0.021695035, 0.004066183, -0.019038767, -0.009185715, -0.04794367, -0.023003541, 0.038234554, -0.021093123, -0.020177167, 0.064483196, -0.017350795, -0.014825377, -0.01672271, -0.052837487, -0.04705389, -0.029284371, 0.022624075, -0.037266262

In [0]:
client = AzureOpenAI(
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
)
spark.sql("DROP TABLE IF EXISTS processed_data.semantic_embeddings")

semantic_embedding_generator = GenerateEmbeddings(
    client=client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    chunk_table_name="processed_data.semantic_chunks"
)
semantic_embedding_generator.embed()
semantic_embedding_generator.save_to_table("processed_data.semantic_embeddings")

display(spark.sql("SELECT chunk_text, embedding FROM processed_data.semantic_embeddings"))

chunk_text embedding Demystifying and Enhancing the Efficiency of Large
TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1,
{yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl. nankai List(0.019352224, -0.0219787, 0.056830022, 0.023248646, 0.015874308, 0.020564444, -0.031286817, 0.043149255, -0.0415041, -0.033884432, 0.043351293, 0.015282629, -0.066094846, -0.013853941, 0.0033841138, 0.03578935, -0.01613407, -0.028184112, 0.015845446, 0.018875996, 0.033942156, -0.009091649, 0.038271513, -0.010217282, 0.02402793, -0.015499097, 0.015946465, 0.018861564, 0.031113643, -0.0504803, 0.022613673, -0.04467896, 0.0015360197, -0.031835202, -0.038300376, 0.078447945, 0.044043988, 0.02171894, -0.004697352, 0.001424178, 0.020290252, 0.0106502175, -0.01759162, -0.030594122, 0.039483733, 0.010671864, -0.012915914, 0.0062018037, 0.0051194644, 0.022469362, 0.005570439, -0.009235961, -0.0018471922, -0.064594, -0.060957342, -0.0034779166, -4.2030838E-4, 0.014835263, 0.024071224, -0.005105033, -0.011884084, -0.027982075, 0.017086528, 0.05850404, -0.021791095, -0.02840058, -0.04066709, -0.004426767, -0.010837823, -0.014907419, -0.047622923, 5.840122E-4, 0.015441372, 0.002864591, 0.06315088, -0.016927784, -0.04805586, 0.009214315, -0.018544078, -9.488507E-4, 0.02330637, -0.002830317, 0.024359846, -0.018111141, -0.0047731157, -0.053828336, -0.05547349, -0.02027582, -0.028588185, -0.006194588, -0.009834856, 0.041330926, -0.043639917, 0.03371126, 0.015022867, 0.0062595285, -0.0011590049, 0.018385334, 0.06297771, 0.024821645, 0.038156062, -0.00697748, 0.06442083, 0.0089906305, 0.0044195517, -0.011292405, -0.054636482, -0.031864066, -0.045198485, 0.03365353, -0.06632575, -0.008702007, 0.013781786, -0.03376898, 0.01092441, -0.033942156, 0.0013087285, 0.016437124, -0.03983008, 0.030507533, 0.022021994, 0.026423506, 0.009640034, 1.5614998E-4, -0.018890426, -0.038156062, 0.01930893, -0.036135696, 5.461303E-4, 0.003559092, 0.03495234, -0.03189293, 0.024763921, -0.0253556, -0.02741926, -0.023537269, -0.019698573, 0.06921198, 0.024735058, 0.014142565, 0.050191674, -0.018616233, 0.020117078, 0.04401513, -0.02417224, 0.0036041895, -0.008997846, -0.016437124, -0.010145126, 0.00971219, 8.9789054E-4, -0.014431189, -0.03486575, -0.033220597, 0.019799592, -0.025182424, 0.0049354667, 0.050076228, 0.015744427, -0.017317427, 0.056050736, -0.02404236, -0.046930227, 0.015903171, 0.025889553, 0.03558731, -0.01746174, -0.016249519, -0.066094846, -0.02694303, 0.013334419, -0.06494035, 0.003364271, 0.0048488793, 0.039223973, -0.023089902, -0.01098935, -0.03581821, -0.04721885, -0.00733826, 0.02258481, 0.006475996, 0.019092463, -0.01650928, -0.038704447, -0.013125166, 0.00434018, 0.025918415, 0.017634913, 0.022021994, -0.039801218, 0.06240046, 0.0298437, 0.009120512, -0.018255454, -0.013125166, 0.023450682, -0.019568693, -0.01855851, 0.011494442, 0.05870608, 0.004625196, 0.025182424, -0.019337794, 0.010484259, -0.049556702, 0.0034382308, 0.0027888273, -0.031604305, -0.0298437, 0.013853941, 0.015672272, 0.023883618, 5.619144E-4, -0.013016933, -0.0075691584, -0.061245967, 0.032066103, -0.018140005, 0.042918358, -0.016913354, 0.012381961, 0.013846726, 0.0036943844, -0.012273726, 0.039137386, 0.024186673, -0.058215417, 0.059023563, -0.040349606, 0.04548711, 0.0028681988, -0.0020907186, 0.007929939, -0.008550479, -0.01855851, -0.0093081165, -0.029930286, 0.0027329065, -0.033191733, -0.023320802, 0.019438813, -0.022772416, -0.0013655513, 0.009235961, 0.025918415, -0.03798289, -0.019409949, 0.045660283, -0.01305301, -0.0074032, -0.036482047, -0.056137327, -0.028429443, -0.04222566, -0.034173056, -0.03783858, 0.042052485, 0.02620704, 0.04782496, -0.0012699446, -0.031835202, -0.025629792, -0.023219783, -0.038444687, -0.0067177187, 0.026409077, 0.07533081, -0.009207099, -0.007973232, 0.038300376, -0.0030323537, -0.027736746, -0.0076990393, -0.033105146, -0.014683735, 0.021213848, 0.0473343, -0.02524015, -0.0035663075, -0.02461961, 0.008557695, 0.033480357, -0.034201916, -0.027837764, -0.02

### Task 4: Simulating Vector Search

In [0]:
class VectorSearch:
    def __init__(self, client, model_name, embeddings_table):
        self.client = client
        self.model_name = model_name
        self.embeddings_table = embeddings_table

    def cosine_similarity(self, vec1, vec2):
        return 1 - cosine(vec1, vec2)

    def embed_query(self, query):
        response = self.client.embeddings.create(input=query, model=self.model_name)
        return response.data[0].embedding

    def search_vectors(self, query, top_k=3):
        query_embedding = self.embed_query(query)
        embeddings_df = spark.sql(f"SELECT chunk_text, embedding FROM {self.embeddings_table}")
        
        results = []
        for row in embeddings_df.toLocalIterator():
            chunk_text = row['chunk_text']
            embedding = row['embedding']
            similarity = self.cosine_similarity(query_embedding, embedding)
            results.append((similarity, chunk_text))
        results.sort(reverse=True, key=lambda x: x[0])
        top_results = results[:top_k]
        schema = StructType([
            StructField("similarity", FloatType(), True),
            StructField("chunk_text", StringType(), True)
        ])
        return spark.createDataFrame(top_results, schema=schema)

In [0]:
client = AzureOpenAI(
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
)
vs_fixed = VectorSearch(
    client=client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    embeddings_table="processed_data.fixed_embeddings"
)
results_1 = vs_fixed.search_vectors("What is SearchAgent-X?")
display(results_1)


similarity,chunk_text
0.6493444,"archAgent-X,ahigh-efficiencyinferenceframeworkforLLM- basedsearchagents. SearchAgent-Xleverageshigh-recallapproximateretrieval and incorporates two key techniques: priority-aware scheduling and non-stall retrieval. ExtensiveexperimentsdemonstratethatSearchAgent-Xconsistently outperforms state-of-the-art systems such as vLLM and HNSW-based retrieval across diverse tasks, achieving up to 3.4× higher throughput and 5× lower la- tency,withoutcompromisinggenerationquality. SearchAgent-Xisavailableat https://gith"
0.51760685,"onquality. SearchAgent-Xisavailableat https://github.com/tiannuo-yang/SearchAgent-X. TraditionalRetrieval-AugmentedGeneration(RAG)typicallyusesasequentialretrieve-then-generate approach [1–8], which limits dynamic interaction with knowledge bases. Recent advancements have ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong reasoningcapabilitiesofLargeLanguageModels(LLMs),allowingforthedynamicandadaptive interleavingofreasoningstepswithretrievalcallsthroughoutthegenerationpr"
0.46781823,"that both highly accurate and overly approximate retrieval methods de- gradesystemefficiency: exactsearchincurssignificantretrievaloverhead,while coarse retrieval requires additional reasoning steps during generation. Second, we identify inefficiencies in system design, including improper scheduling and frequentretrievalstalls,whichleadtocascadinglatency—whereevenminorde- laysinretrievalamplifyend-to-endinferencetime. Toaddressthesechallenges, weintroduceSearchAgent-X,ahigh-efficiencyinferenceframeworkforL"


In [0]:
vs_semantic = VectorSearch(
    client=client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    embeddings_table="processed_data.semantic_embeddings"
)
results_2 = vs_semantic.search_vectors("What is SearchAgent-X?")
display(results_2)

similarity,chunk_text
0.6485054,SearchAgent-Xisavailableat https://github
0.6123781,"0, known as Search Agents [9–16]"
0.5578838,"ExtensiveexperimentsdemonstratethatSearchAgent-Xconsistently outperforms state-of-the-art systems such as vLLM and HNSW-based retrieval across diverse tasks, achieving up to 3"


Analysis of Fixed vs Semantic Chunking Based on Vector Search Results

Fixed Chunking:

	•	Similarity Scores:
	•	Highest: 0.6493 (Chunk 1)
	•	Lowest: 0.4678 (Chunk 3)
	•	Observations: Fixed chunking splits the text arbitrarily, leading to fragmented context. Chunks may cut off in the middle of sentences, causing lower similarity. The fragmented chunks reduce the relevance of search results.

Semantic Chunking:

	•	Similarity Scores:
	•	Highest: 0.6485 (Chunk 1)
	•	Lowest: 0.5579 (Chunk 3)
	•	Observations: Semantic chunking splits content based on meaningful segments, preserving the context. This results in higher similarity scores as chunks are more coherent and relevant to the query.

Key Differences:

	1.	Context Preservation:

	•	Fixed Chunking leads to fragmented and less relevant chunks, lowering similarity.
	•	Semantic Chunking preserves coherent context, resulting in higher similarity.
	2.	Relevance:

	•	Fixed Chunking produces large chunks that lose meaning, affecting search accuracy.
	•	Semantic Chunking generates smaller, contextually rich chunks, leading to more accurate context retrieval.

Conclusion:

	•	Semantic Chunking gives more relevant and accurate results by preserving context, while Fixed Chunking produces fragmented chunks that reduce search accuracy. Semantic chunking is preferred for better vector search results.

### Task 5: RAG Architecture

In [0]:
api_key = os.environ.get("AZURE_OPENAI_API_KEY")
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
deployment = os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.environ.get("AZURE_OPENAI_API_VERSION")

# Initialize the Azure OpenAI LLM client
llm = AzureChatOpenAI(
    azure_endpoint=endpoint,
    azure_deployment=deployment,
    api_version=api_version,
    api_key=api_key,
    temperature=0.0,
)

### TO-DO
system_message = SystemMessagePromptTemplate.from_template(
    "Answer the following question using the provided context below.\n\n"
    "Context:\n{context}\n\n"
    "If the context is insufficient, respond with 'I do not know'."
)

### TO-DO
human_message = HumanMessagePromptTemplate.from_template("{question}")


### TO-DO
chat_prompt = ChatPromptTemplate.from_messages([system_message, human_message])


class RAGPipeline:
    def __init__(self, llm, chat_prompt, vector_search):
        self.llm = llm
        self.chat_prompt = chat_prompt
        self.vector_search = vector_search


    def get_context(self, question):
        results = self.vector_search.search_vectors(question, top_k=3)
        rows = results.collect()
        context = "\n\n".join([str(row['chunk_text']) for row in rows])
        return context

    def invoke(self, question):
        context = self.get_context(question)
        messages = self.chat_prompt.format_messages(question=question, context=context)
        response = self.llm.invoke(messages)
        return response.content

    def invoke_w_lcel(self, question):
        context = self.get_context(question)
        chain = self.chat_prompt | self.llm | StrOutputParser()

        response = chain.invoke({"question": question, "context": context})
        return response

In [0]:
rag_fixed = RAGPipeline(llm, chat_prompt, vs_fixed)
query = "What is SearchAgent-X?"
answer = rag_fixed.invoke(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

print()
answer = rag_fixed.invoke_w_lcel(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is SearchAgent-X?
Answer: SearchAgent-X is a high-efficiency inference framework designed for large language model (LLM)-based search agents. It leverages high-recall approximate retrieval and incorporates two key techniques: priority-aware scheduling and non-stall retrieval. This framework addresses inefficiencies found in traditional retrieval-augmented generation systems, such as significant retrieval overhead from exact search and additional reasoning steps required by coarse retrieval methods. SearchAgent-X improves system efficiency by reducing cascading latency caused by improper scheduling and frequent retrieval stalls. Extensive experiments show that SearchAgent-X consistently outperforms state-of-the-art systems like vLLM and HNSW-based retrieval across diverse tasks, achieving up to 3.4 times higher throughput and 5 times lower latency without compromising generation quality.

Question: What is SearchAgent-X?
Answer: SearchAgent-X is a high-efficiency inferenc

In [0]:

rag_semantic = RAGPipeline(llm, chat_prompt, vs_semantic)
query = "What is SearchAgent-X?"
answer = rag_semantic.invoke(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

print()

answer = rag_semantic.invoke_w_lcel(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is SearchAgent-X?
Answer: SearchAgent-X is a search agent system that has been demonstrated through extensive experiments to consistently outperform state-of-the-art systems such as vLLM and HNSW-based retrieval across diverse tasks, achieving up to 3 times better performance. It is available at https://github.

Question: What is SearchAgent-X?
Answer: SearchAgent-X is a search agent system that has been demonstrated through extensive experiments to consistently outperform state-of-the-art systems such as vLLM and HNSW-based retrieval across diverse tasks, achieving up to 3 times better performance. It is available at https://github.


## Part 2: Databricks Managed Vector Search and Agentic RAG Flow

### Task 1: Data Preparation

In [0]:
import os
from typing import TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from databricks.sdk import WorkspaceClient
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from databricks.vector_search.client import VectorSearchClient
from dotenv import load_dotenv
from pyspark.sql import functions as F
from pyspark.sql import SparkSession


load_dotenv()

True

In [0]:
spark.sql("SELECT version()").show()

+--------------------+
|           version()|
+--------------------+
|4.1.0 00000000000...|
+--------------------+



In [0]:
fixed_chunks_df = spark.read.format("delta").table("processed_data.fixed_chunks")
prepared_df = fixed_chunks_df.select(
    "chunk_text", 
    F.expr("uuid()").alias("Chunk_Id") 
)
spark.sql("DROP TABLE IF EXISTS processed_data.fixed_chunks_prepared")
prepared_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("processed_data.fixed_chunks_prepared")
table_properties = spark.sql("DESCRIBE DETAIL processed_data.fixed_chunks_prepared")
table_properties.show(truncate=False)
data_df = spark.read.table("processed_data.fixed_chunks_prepared")
data_df.show(10, truncate=False) 

+------+------------------------------------+----------------------------------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                                          |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                    |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                                                     |clusterByAuto|
+------+------------------------------------+----------------------------------------------+-----------+--

In [0]:
if 'pa3_catalog' not in schemas_list:
    spark.sql("CREATE SCHEMA IF NOT EXISTS pa3_catalog")
spark.sql("DROP TABLE IF EXISTS pa3_catalog.pa3_schema.fixed_chunks_prepared")
spark.sql("CREATE SCHEMA IF NOT EXISTS pa3_catalog.pa3_schema")
df_stream = spark.readStream.format("delta").table("processed_data.fixed_chunks")
prepared_stream_df = df_stream.select(
    "chunk_text", 
    F.expr("uuid()").alias("Chunk_Id")
)
spark.sql("CREATE VOLUME IF NOT EXISTS pa3_catalog.pa3_schema.checkpoints")
spark.sql("DROP TABLE IF EXISTS pa3_catalog.pa3_schema.fixed_chunks_prepared")
query = prepared_stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "/Volumes/pa3_catalog/pa3_schema/checkpoints/fixed_chunks_prepared") \
    .toTable("pa3_catalog.pa3_schema.fixed_chunks_prepared")
query.awaitTermination()
spark.sql("""
    ALTER TABLE pa3_catalog.pa3_schema.fixed_chunks_prepared
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
display(spark.table("pa3_catalog.pa3_schema.fixed_chunks_prepared"))

chunk_text,Chunk_Id
"Demystifying and Enhancing the Efficiency of Large TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1, {yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn LargeLanguageModel(LLM)-basedsearchagentshaveshownremarkableca- pabilitiesinsolvingcomplextasksbydynamicallydecomposingproblemsand addressing them through interleaved reasoning and retrieval. However, this in- terleaved paradigm introduces substantial efficiency bottlenecks. First, we ob- serve that both highly accurate and overly approximate",b31d17d8-3172-4f69-b936-2de6814c7680
"that both highly accurate and overly approximate retrieval methods de- gradesystemefficiency: exactsearchincurssignificantretrievaloverhead,while coarse retrieval requires additional reasoning steps during generation. Second, we identify inefficiencies in system design, including improper scheduling and frequentretrievalstalls,whichleadtocascadinglatency—whereevenminorde- laysinretrievalamplifyend-to-endinferencetime. Toaddressthesechallenges, weintroduceSearchAgent-X,ahigh-efficiencyinferenceframeworkforL",e3cc67dd-f020-4d92-ab1e-0dbb000aa269
"archAgent-X,ahigh-efficiencyinferenceframeworkforLLM- basedsearchagents. SearchAgent-Xleverageshigh-recallapproximateretrieval and incorporates two key techniques: priority-aware scheduling and non-stall retrieval. ExtensiveexperimentsdemonstratethatSearchAgent-Xconsistently outperforms state-of-the-art systems such as vLLM and HNSW-based retrieval across diverse tasks, achieving up to 3.4× higher throughput and 5× lower la- tency,withoutcompromisinggenerationquality. SearchAgent-Xisavailableat https://gith",d1e0542a-6ec2-4926-b314-6f7169c4d34f
"onquality. SearchAgent-Xisavailableat https://github.com/tiannuo-yang/SearchAgent-X. TraditionalRetrieval-AugmentedGeneration(RAG)typicallyusesasequentialretrieve-then-generate approach [1–8], which limits dynamic interaction with knowledge bases. Recent advancements have ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong reasoningcapabilitiesofLargeLanguageModels(LLMs),allowingforthedynamicandadaptive interleavingofreasoningstepswithretrievalcallsthroughoutthegenerationpr",7919eccf-97a4-476a-9d6a-b2a35589aaee
"ngstepswithretrievalcallsthroughoutthegenerationprocess. Insteadofa fixedpipeline,searchagentscandecidewhenandwhattoretrievebasedonLLM’songoingreasoning,",b42ec813-d077-4e5f-ac53-4323be235e39


In [0]:
spark.sql("DESCRIBE processed_data.fixed_chunks_prepared").show(truncate=False)

+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|chunk_text|string   |NULL   |
|Chunk_Id  |string   |NULL   |
+----------+---------+-------+



### Task 2: Vector Search Management

In [0]:
# FUNCTIONS I MADE IN CASE YOU GUYS NEED TO CLEANUP YOUR WORKSPACE

def vector_store_endpoint_cleanup(vsc):
    endpoints = vsc.list_endpoints().get('endpoints', [])

    if not endpoints:
        print("No endpoints found. Workspace is clear.")
    else:
        for e in endpoints:
            name = e['name']
            print(f"Deleting endpoint: {name}...")
            vsc.delete_endpoint(name)
        
        print("Waiting for workspace quota to refresh...")
        while len(vsc.list_endpoints().get('endpoints', [])) > 0:
            print("Endpoint still terminating... waiting 5s", end="\r")
            time.sleep(5)
        print("\nWorkspace cleared. Quota reset to 0/1.")

def vector_store_index_cleanup(vsc, endpoint_name, index_full_name):
    """Delete the vector store endpoint if it exists"""
    try:
        print(f"Attempting to delete index: {index_full_name}...")
        vsc.delete_index(endpoint_name=endpoint_name, index_name=index_full_name)
        print("Index deletion command sent.")
    except Exception as e:
        print(f"Index not found or already deleted: {e}")

    try:
        print(f"Deleting endpoint: {endpoint_name}...")
        vsc.delete_endpoint(name=endpoint_name)
    except Exception as e:
        print(f"Endpoint not found or already deleted: {e}")

    print("Waiting for Unity Catalog to clear metadata...waiting 5s")
    time.sleep(5)
    print("System cleared. You can now run the creation script.")


In [0]:

DATABRICKS_HOST = os.getenv("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")
client = WorkspaceClient(
    host=DATABRICKS_HOST,
    token=DATABRICKS_TOKEN
)
print("Connected to Databricks Workspace ")

Connected to Databricks Workspace 


In [0]:
vsc = VectorSearchClient()
roll_number = "25280065"
endpoint_name = f"{roll_number}_vector_endpoint"
index_name = "pa3_catalog.vector_index.fixed_vector_index" 

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
# vector_store_endpoint_cleanup(vsc)
# vector_store_index_cleanup(vsc, endpoint_name, index_name)

No endpoints found. Workspace is clear.
Attempting to delete index: pa3_catalog.vector_index.fixed_vector_index...
Index not found or already deleted: Unity Catalog entity pa3_catalog.vector_index.fixed_vector_index does not exist.
Deleting endpoint: 25280065_vector_endpoint...
Endpoint not found or already deleted: Vector search endpoint 25280065_vector_endpoint not found.
Waiting for Unity Catalog to clear metadata...waiting 5s
System cleared. You can now run the creation script.


In [0]:
print(f"Creating Endpoint: {endpoint_name}...")
vsc.create_endpoint(
    name=endpoint_name,
    endpoint_type="STANDARD",
)
### TO-DO: Create the vector store endpoint

print(f"\nCreating index: {index_name}...")
endpoint = vsc.get_endpoint(name=endpoint_name)
vsc.create_delta_sync_index(
    endpoint_name=endpoint_name,
    index_name=index_name,
    pipeline_type="TRIGGERED",
    source_table_name="pa3_catalog.pa3_schema.fixed_chunks_prepared",  
    primary_key="Chunk_Id",                                 
    embedding_source_column="chunk_text",            
    embedding_model_endpoint_name="databricks-gte-large-en",
)

# TO-DO: Create the vector store index

print("Index creation initiated.")


Creating Endpoint: 25280065_vector_endpoint...


---------------------------------------------------------------------------
BadRequest                                Traceback (most recent call last)
File <command-5396045062785531>, line 2
      1 print(f"Creating Endpoint: {endpoint_name}...")
----> 2 vsc.create_endpoint(
      3     name=endpoint_name,
      4     endpoint_type="STANDARD",
      5 )
      6 ### TO-DO: Create the vector store endpoint
      8 print(f"\nCreating index: {index_name}...")

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-258bd91b-1eec-4ceb-bc4d-7e838d4b6970/lib/python3.12/site-packages/databricks/vector_search/client.py:302, in VectorSearchClient.create_endpoint(self, name, endpoint_type, budget_policy_id, usage_policy_id, min_qps)
    299 if min_qps is not None:
    300     json_data["min_qps"] = min_qps
--> 302 return RequestUtils.issue_request(
    303     url=f"{self.workspace_url}/api/2.0/vector-search/endpoints",
    304     token=self._get_token_for_request(),
    305     method="POST",
    306 

In [0]:
%sql
SELECT * FROM pa3_catalog.pa3_schema.fixed_chunks_prepared LIMIT 10;

chunk_text,Chunk_Id
"Demystifying and Enhancing the Efficiency of Large TiannuoYang1∗,ZebinYao1∗,BowenJin2,LixiaoCui1,YusenLi1, {yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn LargeLanguageModel(LLM)-basedsearchagentshaveshownremarkableca- pabilitiesinsolvingcomplextasksbydynamicallydecomposingproblemsand addressing them through interleaved reasoning and retrieval. However, this in- terleaved paradigm introduces substantial efficiency bottlenecks. First, we ob- serve that both highly accurate and overly approximate",b31d17d8-3172-4f69-b936-2de6814c7680
"that both highly accurate and overly approximate retrieval methods de- gradesystemefficiency: exactsearchincurssignificantretrievaloverhead,while coarse retrieval requires additional reasoning steps during generation. Second, we identify inefficiencies in system design, including improper scheduling and frequentretrievalstalls,whichleadtocascadinglatency—whereevenminorde- laysinretrievalamplifyend-to-endinferencetime. Toaddressthesechallenges, weintroduceSearchAgent-X,ahigh-efficiencyinferenceframeworkforL",e3cc67dd-f020-4d92-ab1e-0dbb000aa269
"archAgent-X,ahigh-efficiencyinferenceframeworkforLLM- basedsearchagents. SearchAgent-Xleverageshigh-recallapproximateretrieval and incorporates two key techniques: priority-aware scheduling and non-stall retrieval. ExtensiveexperimentsdemonstratethatSearchAgent-Xconsistently outperforms state-of-the-art systems such as vLLM and HNSW-based retrieval across diverse tasks, achieving up to 3.4× higher throughput and 5× lower la- tency,withoutcompromisinggenerationquality. SearchAgent-Xisavailableat https://gith",d1e0542a-6ec2-4926-b314-6f7169c4d34f
"onquality. SearchAgent-Xisavailableat https://github.com/tiannuo-yang/SearchAgent-X. TraditionalRetrieval-AugmentedGeneration(RAG)typicallyusesasequentialretrieve-then-generate approach [1–8], which limits dynamic interaction with knowledge bases. Recent advancements have ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong reasoningcapabilitiesofLargeLanguageModels(LLMs),allowingforthedynamicandadaptive interleavingofreasoningstepswithretrievalcallsthroughoutthegenerationpr",7919eccf-97a4-476a-9d6a-b2a35589aaee
"ngstepswithretrievalcallsthroughoutthegenerationprocess. Insteadofa fixedpipeline,searchagentscandecidewhenandwhattoretrievebasedonLLM’songoingreasoning,",b42ec813-d077-4e5f-ac53-4323be235e39


### Task 3: Adaptive RAG Workflow

In [0]:
# Check if index exists
index = vsc.get_index(
    endpoint_name=endpoint_name,
    index_name="pa3_catalog.vector_index.fixed_vector_index"
)
print(index)
index = vsc.get_index(
    endpoint_name=endpoint_name,
    index_name="pa3_catalog.vector_index.fixed_vector_index"
)
print(index.describe()["status"])

{'detailed_state': 'ONLINE_NO_PENDING_UPDATE', 'message': 'Index creation succeeded. Check latest status: https://dbc-f40c52f8-2213.cloud.databricks.com/explore/data/pa3_catalog/vector_index/fixed_vector_index', 'indexed_row_count': 5, 'triggered_update_status': {'last_processed_commit_version': 2, 'last_processed_commit_timestamp': '2026-04-21T18:49:04Z'}, 'ready': True, 'index_url': 'dbc-f40c52f8-2213.cloud.databricks.com/api/2.0/vector-search/indexes/pa3_catalog.vector_index.fixed_vector_index'}


In [0]:
index = vsc.get_index(
    endpoint_name=endpoint_name,
    index_name="pa3_catalog.vector_index.fixed_vector_index" 
)

while True:
    status = index.describe().get("status", {})
    detailed_state = status.get("detailed_state")
    ready = status.get("ready", False)
    print(f" State: {detailed_state}")
    if ready:
        print(" Index is ready!")
        break
    time.sleep(30)

⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_ENDPOINT
⏳ State: PROVISIONING_INDEX
⏳ State: PROVISIONING_PIPELINE_RESOURCES
⏳ State: PROVISIONING_INITIAL_SNAPSHOT
⏳ State: ONLINE_TRIGGERED_UPDATE
✅ Index is ready!


In [0]:
llm = AzureChatOpenAI(
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"), 
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"), 
    temperature=0.0
)

class GraphState(TypedDict):
    question: str
    initial_answer: str
    grade: str
    final_answer: str
    context: str

In [0]:
def retrieve_formatted_context(question: str, topk: int = 3):
    index = VectorSearchClient(disable_notice=True).get_index(
        endpoint_name="25280065_vector_endpoint",
        index_name="pa3_catalog.vector_index.fixed_vector_index"
    )
    results = index.similarity_search(
        query_text=question,
        columns=["chunk_text"],
        num_results=topk
    )
    print(f"[retriever] Raw results: {results}")
    rows = results.get("result", {}).get("data_array", [])
    print(f"[retriever] Rows found: {len(rows)}")
    formatted_context = "\n".join([row[0] for row in rows if row])
    return formatted_context

### TO-DO
context_retriever_node = retrieve_formatted_context

### TO-DO
system_message = SystemMessagePromptTemplate.from_template(
    "You are a helpful assistant. Use only the provided context to answer the question. "
    "If the context does not contain the answer, say 'I do not know'."
)

### TO-DO
human_message = HumanMessagePromptTemplate.from_template(
    "Here is the context: {context}\nAnswer the following question: {question}"
)

### TO-DO
chat_prompt = ChatPromptTemplate.from_messages([system_message, human_message])

### TO-DO
rag_chain = chat_prompt | llm | StrOutputParser()
def base_rag_node(state: GraphState) -> dict:
    question = state.get("question")
    if not question:
        raise ValueError("No question provided in the state!")
    context = context_retriever_node(question)
    final_answer = rag_chain.invoke({"question": question, "context": context})
    state["context"] = context   # ← store context in state
    state["final_answer"] = final_answer
    return state




In [0]:
### TO-DO
class AnswerGrader(BaseModel):
    binary_score: str = Field(description="Relevance score: '0' or '1'")

### TO-DO
grader_system_message = SystemMessagePromptTemplate.from_template(
    "You are an assistant that checks if a given context contains enough information to answer a question. "
    "Reply with only 'relevant' or 'not relevant'."
)
### TO-DO
grader_human_message = HumanMessagePromptTemplate.from_template(
    "Context: {context}\nQuestion: {question}\n"
    "Does the context contain information to answer the question? Reply 'relevant' or 'not relevant'."
)

### TO-DO
grader_prompt = ChatPromptTemplate.from_messages([grader_system_message, grader_human_message])

### TO-DO
grader_chain = grader_prompt | llm | StrOutputParser()


def grader_node(state: GraphState) -> dict:
    question = state.get("question")
    context = context_retriever_node(question)
    evaluation_result = grader_chain.invoke({"question": question, "context": context})
    state["grade"] = "relevant" if "relevant" in evaluation_result.lower() and "not" not in evaluation_result.lower() else "not relevant"
    return state


In [0]:
### TO-DO
class CategoryRouter(BaseModel):
    category: str = Field(..., description="The category of the question (e.g., 'azure', 'databricks')")


### TO-DO
router_prompt = ChatPromptTemplate.from_template(
    "You must respond with exactly one word only.\n"
    "If the question is about Azure respond with: azure\n"
    "If the question is about Databricks respond with: databricks\n"
    "No punctuation, no explanation, no other words.\n"
    "Question: {question}\n"
    "One word answer:"
)

### TO-DO
router_chain = router_prompt | llm | StrOutputParser()


def text_file_fallback_node(state: GraphState) -> dict:
    raw_category = router_chain.invoke({"question": state["question"]}).strip().lower()
    if "databricks" in raw_category:
        category = "databricks"
    elif "azure" in raw_category:
        category = "azure"
    else:
        category = None

    print(f"[fallback_node] Parsed category: {category}")

    file_map = {
        "databricks": "databricks_info.txt",
        "azure": "azure_info.txt",
    }
    file_path = file_map.get(category, None)
    print(f"[fallback_node] File path: {file_path}")

    if file_path is None:
        return {**state, "final_answer": f"Category not found: {raw_category}"}
    try:
        with open(file_path, "r") as f:
            full_text = f.read()
        fallback_chain = ChatPromptTemplate.from_template(
            "Answer the question using the context below.\nContext: {context}\nQuestion: {question}"
        ) | llm | StrOutputParser()
        response = fallback_chain.invoke({"question": state["question"], "context": full_text})
        print(f"[fallback_node] Response: {response[:200]}")
        state["final_answer"] = response
        return state
    except Exception as e:
        print(f"[fallback_node] ERROR: {e}")
        return {**state, "final_answer": f"Fallback failed. Error: {e}"}


### Task 4: Creating the Agentic RAG Graph

In [0]:


def should_fallback(state):
    if state.get("grade") != "relevant":
        return "file_fallback_node"
    return END

workflow = StateGraph(state_schema=GraphState)
workflow.add_node("base_rag_node", base_rag_node)
workflow.add_node("grader_node", grader_node)
workflow.add_node("file_fallback_node", text_file_fallback_node)
workflow.add_edge(START, "base_rag_node")
workflow.add_edge("base_rag_node", "grader_node")
workflow.add_conditional_edges(
    "grader_node",
    should_fallback,
    {
        "file_fallback_node": "file_fallback_node",
        END: END
    }
)

workflow.add_edge("file_fallback_node", END)

compiled_adaptive_rag = workflow.compile()

### Task 5: Executing tbe Agentic RAG FLow

In [0]:
def agentic_rag_flow(question: str):
    print(f"--- STARTING ADAPTIVE RAG FLOW ---")
    print(f"User Question: {question}\n")
    state: GraphState = {
        "question": question,
        "initial_answer": "",
        "grade": "",
        "final_answer": "",
        "context": ""
    }
    print(f"--- Executing Base RAG Node ---")
    state = base_rag_node(state)
    print(f"Updated State: {state}\n")

    print(f"--- Executing Grader Node ---")
    state = grader_node(state)
    print(f"Grade: {state['grade']}\n")

    if state["grade"] != "relevant":
        print(f"--- Answer not relevant, invoking fallback ---")
        state = text_file_fallback_node(state)
    else:
        print(f"--- Answer is relevant, skipping fallback ---")

    print(f"--- Flow Completed ---")
    return state["final_answer"]

In [0]:
try:
    print("TEST 1: calling function")
    final_state = agentic_rag_flow("What is Databricks?")
    print(f"TEST 2: function returned: '{final_state}'")
except Exception as e:
    import traceback
    traceback.print_exc()

TEST 1: calling function
--- STARTING ADAPTIVE RAG FLOW ---
User Question: What is Databricks?

--- Executing Base RAG Node ---
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[retriever] Raw results: {'manifest': {'column_count': 2, 'columns': [{'name': 'chunk_text'}, {'name': 'score'}]}, 'result': {'row_count': 3, 'data_array': [['onquality. SearchAgent-Xisavailableat\nhttps://github.com/tiannuo-yang/SearchAgent-X.\nTraditionalRetrieval-AugmentedGeneration(RAG)typicallyusesasequentialretrieve-then-generate\napproach [1–8], which limits dynamic interaction with knowledge bases. Recent advancements\nhave ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong\nreasoningcapabilitiesofLargeLanguageModels(LLMs),allowingforthedynamicandadaptive\ninterleavingofreasoningstepswithretrievalcallsthroughout

## `PART 3: Model Context Protocol (MCP) Integration`

In [0]:
%pip install --upgrade mlflow[databricks] databricks-agents
dbutils.library.restartPython()
%pip install langchain --upgrade

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import time
import yaml
import mlflow
from dotenv import load_dotenv
from mlflow.pyfunc import ResponsesAgent
from typing import Any, Dict
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain.tools import tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from databricks.vector_search.client import VectorSearchClient
load_dotenv()

True

In [0]:
@tool
def vector_search_tool(query: str) -> str:
    """Searches the Databricks Vector Search index for chunks relevant to the query and returns formatted results."""
    try:
        index = VectorSearchClient(disable_notice=True).get_index(
            endpoint_name="25280065_vector_endpoint",
            index_name="pa3_catalog.vector_index.fixed_vector_index"
        )
        results = index.similarity_search(
            query_text=query,
            columns=["chunk_text"],
            num_results=3
        )
        rows = results.get("result", {}).get("data_array", [])
        if not rows:
            return "No relevant documents found in the vector store."
        formatted = "\n\n".join([
            f"Chunk {i+1}: {row[0]}" for i, row in enumerate(rows) if row
        ])
        return formatted
    except Exception as e:
        return f"Database search failed: {e}"


@tool
def vector_store_endpoint_cleanup(vsc) -> str:
    """Deletes the configured Databricks Vector Search endpoint if it exists and returns a confirmation message."""
    try:
        endpoint_name = os.getenv("VECTOR_ENDPOINT_NAME", "25280065_vector_endpoint")
        existing = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
        if endpoint_name in existing:
            vsc.delete_endpoint(endpoint_name)
            return f"Endpoint '{endpoint_name}' successfully deleted."
        return f"Endpoint '{endpoint_name}' not found — nothing to delete."
    except Exception as e:
        return f"Endpoint cleanup failed: {e}"


@tool
def vector_store_index_cleanup(endpoint_name: str, index_full_name: str) -> str:
    """Deletes the specified Databricks Vector Search index from the given endpoint if it exists and returns a confirmation message."""
    try:
        vsc = VectorSearchClient(disable_notice=True)
        existing_indexes = [
            i["name"] for i in vsc.list_indexes(endpoint_name).get("vector_indexes", [])
        ]
        if index_full_name in existing_indexes:
            vsc.delete_index(endpoint_name=endpoint_name, index_name=index_full_name)
            return f"Index '{index_full_name}' successfully deleted from endpoint '{endpoint_name}'."
        return f"Index '{index_full_name}' not found — nothing to delete."
    except Exception as e:
        return f"Index cleanup failed: {e}"

In [0]:

class VectorSearchAgent(ResponsesAgent):
    def __init__(self, config_path: str):
        try:
            with open(config_path, 'r') as f:
                self.config = yaml.safe_load(f)
        except FileNotFoundError:
            self.config = {}  

        super().__init__()

        self.system_prompt = (
            "You are a precise data engineering assistant. "
            "To answer questions about the course or documentation, you MUST use "
            "the provided vector_search_tool to retrieve context. "
            "If the tool does not return relevant information, state that you do not know."
        )

    def predict(self, payload) -> Dict[str, Any]:
        """Executes the Agent Loop locally."""
        if isinstance(payload, dict):
            messages = payload.get("messages", payload.get("input", []))
        elif isinstance(payload, list):
            messages = payload
        else:
            messages = getattr(payload, "messages", getattr(payload, "input", []))
            
        user_question = messages[-1]['content'] if isinstance(messages[-1], dict) else messages[-1].content
        
        ### TO-DO
        llm = AzureChatOpenAI(
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            temperature=0.0,
        )
        
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", self.system_prompt),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad"),
        ])

        
        tools = [
            vector_search_tool,
            vector_store_index_cleanup,
            vector_store_endpoint_cleanup
        ]
        agent_logic = create_tool_calling_agent(llm, tools, prompt_template)
        agent_executor = AgentExecutor(
            agent=agent_logic,
            tools=tools,
            verbose=True,
            handle_parsing_errors=True,
        )
        
        # RUN THE LOOP
        raw_response = agent_executor.invoke({"input": user_question})["output"]
        
        return {"choices": [{"message": {"role": "assistant", "content": raw_response}}]}

agent = VectorSearchAgent(config_path="agent_config.yml")
mlflow.models.set_model(model=agent)


### Task 4: Executing and Testing

In [0]:
test_questions = [
    "What is the Databricks Vector Search endpoint cleanup function supposed to do?",
    "What tools do you have and what do they do?"
]

print("Starting Local Agent Executor Test...\n" + "="*50)

for question in test_questions:
    print(f"\nUser Query: {question}")
    
    # Required schema structure
    payload = {
        "input": [{"role": "user", "content": question}], 
        "messages": [{"role": "user", "content": question}]
    }
    
    # Invoke the agent
    response = agent.predict(payload)
    final_answer = response["choices"][0]["message"]["content"]
    
    print(f"Final Answer:\n{final_answer}")
    print("-" * 50)

Starting Local Agent Executor Test...

User Query: What is the Databricks Vector Search endpoint cleanup function supposed to do?


> Entering new AgentExecutor chain...

Invoking: `vector_store_endpoint_cleanup` with `{'vsc': {}}`


Endpoint cleanup failed: 'dict' object has no attribute 'list_endpoints'The Databricks Vector Search endpoint cleanup function is intended to delete the configured Databricks Vector Search endpoint if it exists and return a confirmation message. However, it seems there was an error when attempting to perform the cleanup due to a missing or incorrect attribute in the provided input.

> Finished chain.
Final Answer:
The Databricks Vector Search endpoint cleanup function is intended to delete the configured Databricks Vector Search endpoint if it exists and return a confirmation message. However, it seems there was an error when attempting to perform the cleanup due to a missing or incorrect attribute in the provided input.
-----------------------------------

[Trace(trace_id=tr-7c1636431170fc896a1effb4b00e888f), Trace(trace_id=tr-3f8d24891d75fcb0fb10675917fb73b8)]